# can_motor_node — CAN-Connected BLDC Motor-Driver Node

STM32G431CBU6 MCU + DRV8353RS gate-driver + AS5047P magnetic encoder (SPI) + TCAN330GD CAN transceiver + AP63205WU 3.3V buck regulator from 12V input.

In [ ]:
import pathlib
import hw_toolkit as hw

board = hw.Board("can_motor_node")
print(f"Board created: {board.project_id}")

## MCU — STM32G431CBU6 (motor-control class, 170 MHz Cortex-M4)

In [ ]:
# STM32G431CBU6: 128KB flash, FPU, hardware encoder timer, 3x advanced PWM timers
# Handles 6-PWM + SPI encoder + CAN (FDCAN1) all in one package
mcu = board.module(
    id="mcu",
    category="mcu",
    mpn="STM32G431CBU6",
    package="UFQFPN-48",
    price_usd=2.50,
    manufacturer="STMicroelectronics",
)
print(mcu)

## 3.3V Buck Regulator — AP63205WU (fed from 12V)

In [ ]:
# AP63205WU: 600mA synchronous buck, 3.8V–32V input, SOT-23-6
# Models as a board.module with VIN/VOUT/GND pins per spec
reg = board.module(
    id="reg",
    category="buck_regulator",
    mpn="AP63205WU-7",
    package="SOT-23-6",
    price_usd=0.60,
    manufacturer="Diodes Inc",
)
print(reg)

## 12V Input Header — 2-pin connector

In [ ]:
# 2-pin 12V input power connector
pwr_conn = board.module(
    id="pwr_conn",
    category="connector",
    mpn="22-23-2021",
    package="THT-2pin",
    price_usd=0.40,
    manufacturer="Molex",
)
print(pwr_conn)

## 3-Phase Gate Driver — DRV8353RS (SPI-configurable smart driver)

In [ ]:
# DRV8353RS: 3-phase smart gate driver with SPI configuration.
# Accepts 6 PWM inputs (INL_A/B/C + INH_A/B/C) OR 3-PWM mode via SPI.
# SPI interface used for fault status, gate-drive tuning.
# PVDD up to 60V, SPI for config + fault readback.
drv = board.module(
    id="drv",
    category="gate_driver",
    mpn="DRV8353RS",
    package="VQFN-40",
    price_usd=4.50,
    manufacturer="Texas Instruments",
)
print(drv)

## Magnetic Encoder — AS5047P (SPI, 14-bit)

In [ ]:
# AS5047P: 14-bit absolute magnetic rotary encoder, SPI + ABI + UVW outputs
# Used for rotor position feedback
enc = board.module(
    id="enc",
    category="encoder",
    mpn="AS5047P-TS_EK_AB",
    package="TSSOP-14",
    price_usd=3.20,
    manufacturer="ams",
)
print(enc)

## CAN Transceiver — TCAN330GD

In [ ]:
# TCAN330GD: 5 Mbps CAN FD transceiver, 3.3V logic, SOT-23-5
can_xcvr = board.module(
    id="can_xcvr",
    category="can_transceiver",
    mpn="TCAN330GD",
    package="SOT-23-5",
    price_usd=0.80,
    manufacturer="Texas Instruments",
)
print(can_xcvr)

## CAN Bus Connector — 2-pin CANH/CANL

In [ ]:
# 2-pin screw terminal for CAN bus (CANH / CANL)
can_conn = board.module(
    id="can_conn",
    category="connector",
    mpn="691321100002",
    package="THT-2pin",
    price_usd=0.35,
    manufacturer="Wurth",
)
print(can_conn)

## Passives — Decoupling Caps + PWM Gate Resistors

In [ ]:
# 12V rail bulk + decoupling
# Note: subsystem id = lowercase(refdes), e.g. "C1" → id "c1"
c_12v_bulk = board.capacitor("C1", "100u", package="1210")    # 12V bulk cap
c_12v_dec  = board.capacitor("C2", "100n", package="0402")    # 12V HF decoupling

# 3.3V rail decoupling (MCU VDD, encoder, CAN xcvr)
c_3v3_bulk = board.capacitor("C3", "10u",  package="0805")    # 3.3V bulk
c_3v3_dec1 = board.capacitor("C4", "100n", package="0402")    # 3.3V HF MCU
c_3v3_dec2 = board.capacitor("C5", "100n", package="0402")    # 3.3V HF enc
c_3v3_dec3 = board.capacitor("C6", "100n", package="0402")    # 3.3V HF CAN

# Regulator input/output caps
c_reg_in  = board.capacitor("C7", "10u",  package="0805")    # reg VIN local cap
c_reg_out = board.capacitor("C8", "22u",  package="0805")    # reg VOUT

# DRV8353 PVDD decoupling (12V gate drive supply)
c_pvdd_bulk = board.capacitor("C9",  "10u",  package="1206") # PVDD bulk
c_pvdd_dec  = board.capacitor("C10", "100n", package="0402") # PVDD HF

# CAN bus termination resistors (120 Ω end-termination)
r_can_term = board.resistor("R1", "120", package="0402")

# Print actual subsystem IDs (= lowercase refdes)
print("Capacitor ids:", c_12v_bulk.id, c_12v_dec.id,
      c_3v3_bulk.id, c_3v3_dec1.id, c_3v3_dec2.id, c_3v3_dec3.id,
      c_reg_in.id, c_reg_out.id, c_pvdd_bulk.id, c_pvdd_dec.id)
print("Resistor id:", r_can_term.id)
print("Passives defined")

## Power Nets

In [ ]:
# 12V raw supply from input connector
v12  = board.power("v12", voltage_v=12.0)
# 3.3V regulated output
v3v3 = board.power("v3v3", voltage_v=3.3)
# Common ground
gnd  = board.gnd()

# 12V net: input connector → regulator VIN + DRV PVDD + bulk/dec caps
# Subsystem ids = lowercase(refdes): C1=c1, C2=c2, C7=c7, C9=c9, C10=c10
v12 += "pwr_conn.VIN"
v12 += "reg.VIN"
v12 += "drv.PVDD"
v12 += "c1.POS", "c2.POS"       # C1=12V bulk, C2=12V dec
v12 += "c7.POS"                 # C7=reg VIN local
v12 += "c9.POS", "c10.POS"      # C9=PVDD bulk, C10=PVDD dec

# 3.3V net: regulator VOUT → MCU VDD, encoder VDD, CAN xcvr VCC + caps
# C3=3v3 bulk, C4=3v3 MCU dec, C5=3v3 enc dec, C6=3v3 CAN dec, C8=reg out
v3v3 += "reg.VOUT"
v3v3 += "mcu.VDD"
v3v3 += "enc.VDD3V3"
v3v3 += "can_xcvr.VCC"
v3v3 += "c3.POS", "c4.POS", "c5.POS", "c6.POS"  # 3.3V decoupling caps
v3v3 += "c8.POS"                                  # reg output cap

# GND: all components
gnd += "pwr_conn.GND"
gnd += "reg.GND"
gnd += "mcu.GND"
gnd += "drv.GND"
gnd += "enc.GND"
gnd += "can_xcvr.GND"
gnd += "c1.NEG", "c2.NEG"       # 12V caps
gnd += "c3.NEG", "c4.NEG", "c5.NEG", "c6.NEG"  # 3.3V caps
gnd += "c7.NEG", "c8.NEG"       # reg caps
gnd += "c9.NEG", "c10.NEG"      # PVDD caps

print("Power nets wired")
print("v12  members:", len(v12.members))
print("v3v3 members:", len(v3v3.members))
print("gnd  members:", len(gnd.members))

## Signal Nets — SPI (DRV8353 config) + SPI (encoder)

In [ ]:
# Per §6.2: both DRV8353RS and AS5047P share SPI1 (MOSI/MISO/SCK).
# Use board.spi() once for the shared bus — the bundled cs goes to DRV.
# Encoder CS declared as a separate board.signal().
# STM32G431 SPI1: PA5=SCK, PA6=MISO, PA7=MOSI, PA4=NSS(DRV CS)
spi_mosi, spi_miso, spi_sck, drv_cs = board.spi("spi1")

# Shared bus: MCU + DRV + encoder all on MOSI/MISO/SCK
spi_mosi += "mcu.PA7", "drv.SDI",  "enc.MOSI"
spi_miso += "mcu.PA6", "drv.SDO",  "enc.MISO"
spi_sck  += "mcu.PA5", "drv.SCLK", "enc.CLK"

# DRV CS: MCU PA4 → DRV nSCS (uses bundled cs from board.spi())
drv_cs   += "mcu.PA4", "drv.nSCS"

# Encoder CS: separate signal per §6.2
enc_cs = board.signal("enc_cs", protocol="spi")
enc_cs += "mcu.PB12", "enc.CSN"

print("SPI nets wired")

## Signal Nets — 6x PWM (MCU → DRV8353RS gate inputs)

In [ ]:
# 6 PWM signals: MCU TIM1 (advanced timer) → DRV8353RS gate inputs
# STM32G431 TIM1 6-PWM:
#   High-side: PA8(CH1), PA9(CH2), PA10(CH3)
#   Low-side complementary: PB13(CH1N), PB0(CH2N), PB1(CH3N)
# Note: PB13 is shared with SPI1_SCK (PA5 used above) — PB13 only used here for CH1N.
# Since SPI SCK is on PA5, PB13 is free for TIM1_CH1N.

pwm_ah = board.signal("pwm_ah", protocol="pwm")  # Phase A high
pwm_al = board.signal("pwm_al", protocol="pwm")  # Phase A low
pwm_bh = board.signal("pwm_bh", protocol="pwm")  # Phase B high
pwm_bl = board.signal("pwm_bl", protocol="pwm")  # Phase B low
pwm_ch = board.signal("pwm_ch", protocol="pwm")  # Phase C high
pwm_cl = board.signal("pwm_cl", protocol="pwm")  # Phase C low

# MCU TIM1 → DRV8353RS INH/INL pins
pwm_ah += "mcu.PA8",  "drv.INHA"
pwm_al += "mcu.PB13", "drv.INLA"
pwm_bh += "mcu.PA9",  "drv.INHB"
pwm_bl += "mcu.PB0",  "drv.INLB"
pwm_ch += "mcu.PA10", "drv.INHC"
pwm_cl += "mcu.PB1",  "drv.INLC"

# DRV ENABLE: MCU GPIO → DRV ENABLE pin
drv_enable = board.signal("drv_enable", protocol="gpio")
drv_enable += "mcu.PB2", "drv.ENABLE"

# DRV FAULT: DRV nFAULT → MCU interrupt pin
drv_fault = board.signal("drv_fault", protocol="gpio")
drv_fault += "drv.nFAULT", "mcu.PC14"

print("PWM signals wired")

## Signal Nets — CAN Bus

In [ ]:
# CAN bus: MCU FDCAN1 → TCAN330GD → 2-pin connector
# STM32G431 FDCAN1: PB8=RX, PB9=TX
canh, canl = board.can("bus")

# MCU CANTX/CANRX → transceiver TXD/RXD (single-ended logic signals)
can_tx = board.signal("can_tx", protocol="can")
can_rx = board.signal("can_rx", protocol="can")
can_tx += "mcu.PB9",  "can_xcvr.TXD"
can_rx += "mcu.PB8",  "can_xcvr.RXD"

# Transceiver CANH/CANL → differential pair → connector
canh += "can_xcvr.CANH", "can_conn.CANH"
canl += "can_xcvr.CANL", "can_conn.CANL"

# CAN bus termination (120Ω across CANH-CANL)
# R1 refdes → subsystem id "r1"
canh += "r1.A"
canl += "r1.B"

print("CAN bus wired")

## Board Summary

In [ ]:
print(board.summary())

## ERC Check

In [ ]:
board.check_erc(expected_codes=(
    "pin_not_connected",          # unused MCU pins, DRV boot/test pins
    "lib_symbol_issues",          # hwagent lib synthesized at runtime
    "pin_to_pin",                 # power-rail pin ties
    "power_pin_not_driven",       # connector power pins without PWR_FLAG
    "unconnected_wire_endpoint",  # synthesized wire-layout artifact
))
print("ERC passed")

## Export KiCad Project

In [ ]:
import pathlib

out = pathlib.Path("/Users/juanantonioluera/ws/hw-toolkit/docs/projects/can_motor_node/can_motor_node.zip")
board.export_kicad(out, unzip=True)

print(f"Exported: {out}")
print(f"Zip exists: {out.exists()}")

board.show()